# Querying and Filtering Nodes

This notebook demonstrates the query and filtering capabilities of `NetworkXGraph`.

Key methods covered:
- `get_node()` — retrieve a node by internal id
- `get_node_ids()` / `get_nodes()` — list all node ids
- `get_node_pks()` — list all primary keys
- `get_edges()` / `get_edge_attrs()` — list edges and their attributes
- `get_node_attrs()` — inspect node attributes
- `find_nodes_by_property()` — index-based property lookup
- `find_nodes()` — multi-filter search (intersection / union)
- `debug()` / `print_debug()` — human-readable graph snapshot

In [ ]:
import importlib.util

package_to_check = 'drm'
spec = importlib.util.find_spec(package_to_check)

if spec is None:
    print(f'⚠️ {package_to_check} no està instal·lat. Iniciant instal·lació...')
    %pip install -q --upgrade drm-tools
    print("✅ Instal·lació completada. L'estat del kernel PODRIA requerir un reinici.")
else:
    print(f'✅ {package_to_check} ja està present al sistema. Saltant instal·lació.')

In [ ]:
from drm import NetworkXGraph, Node, WeakNode

graph = NetworkXGraph()

# Create a small graph: authors and papers
alice = Node(pk={"name": "Alice"}, main_label="Author")
alice["affiliation"] = "MIT"
alice["year"] = 2020

bob = Node(pk={"name": "Bob"}, main_label="Author")
bob["affiliation"] = "Stanford"
bob["year"] = 2021

carol = Node(pk={"name": "Carol"}, main_label="Author")
carol["affiliation"] = "MIT"
carol["year"] = 2022

graph.insertNode(alice)
graph.insertNode(bob)
graph.insertNode(carol)

# Create papers
paper1 = Node(pk={"title": "Graphs 101"}, main_label="Paper")
paper1["year"] = 2023
paper1["topic"] = "intro"

paper2 = Node(pk={"title": "Advanced Graphs"}, main_label="Paper")
paper2["year"] = 2024
paper2["topic"] = "advanced"

graph.insertNode(paper1)
graph.insertNode(paper2)

# Create relations: Alice wrote paper1, Bob wrote paper1, Carol wrote paper2
from drm import Relation
graph.insertRelation(Relation(alice, paper1, "WROTE"))
graph.insertRelation(Relation(bob, paper1, "WROTE"))
graph.insertRelation(Relation(carol, paper2, "WROTE"))

print("Graph created successfully.")
print("Node ids:", graph.get_node_ids())
print("Edges:", graph.get_edges())

## Retrieving nodes by id

`get_node()` returns a full `Node` object with all its attributes.

In [ ]:
all_ids = graph.get_node_ids()
first_id = all_ids[0]
node = graph.get_node(first_id)
print(f"Node id {first_id}:")
print(f"  main_label: {node.main_label}")
print(f"  pk: {node._primary_key}")
print(f"  labels: {node.labels}")

## Getting node and edge attributes

`get_node_attrs()` and `get_edge_attrs()` return raw attribute dicts.

In [ ]:
# Node attributes
alice_id = graph.checkNode(alice)
attrs = graph.get_node_attrs(alice_id)
print("Alice's attributes:", attrs)

# Edge attributes
edges = graph.get_edges()
for src, dst, rel_type in edges:
    edge_attrs = graph.get_edge_attrs(src, dst, rel_type)
    print(f"  Edge {src} --[{rel_type}]--> {dst}: {edge_attrs}")

## Listing all primary keys

`get_node_pks()` returns a list of `{main_label, pk}` dicts.

In [ ]:
for entry in graph.get_node_pks():
    print(f"  {entry['main_label']:8s} pk={entry['pk']}")

## Property-based filtering

DRM maintains a secondary index on scalar properties. `find_nodes_by_property()` returns
all node ids matching a given property value.

In [ ]:
# Find all authors from MIT
mit_authors = graph.find_nodes_by_property("affiliation", "MIT")
print("MIT authors:", mit_authors)
for nid in mit_authors:
    node = graph.get_node(nid)
    print(f"  - {node._primary_key}")

## Multi-filter search

`find_nodes(filters, match)` supports:
- `match="all"` (intersection) — all filters must match
- `match="any"` (union) — at least one filter must match

In [ ]:
# Find nodes that are from MIT AND wrote a paper (match="all" = intersection)
result = graph.find_nodes({"affiliation": "MIT"}, match="all")
print("MIT nodes:", result)

# Find all papers (topic = "intro" OR topic = "advanced")
result = graph.find_nodes({"topic": "intro"}, match="any")
print("Intro papers:", result)

# No filters returns all nodes
all_nodes = graph.find_nodes({})
print("All nodes:", all_nodes)

## Debug snapshot

`debug()` returns a dict suitable for inspection; `print_debug()` prints it formatted.

In [ ]:
graph.print_debug()

## Cleanup

In [ ]:
graph.close()
print("Graph closed.")